In [ ]:
%%capture
! unzip /content/archive.zip

In [ ]:
import tensorflow as tf
import os
import glob
import matplotlib.pyplot as plt
from tqdm import trange
import numpy as np
import cv2
from google.colab.patches import cv2_imshow

In [ ]:
path_train = "/content/train"
image_files = tf.data.Dataset.list_files(os.path.join(path_train, 'images/*.jpg'))
labels_files = glob.glob(os.path.join(path_train, 'labels/*.txt'))

In [ ]:
def load_image(image_path):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (224, 224))
    image = image / 255.0
    return image

def load_label(label_line):
    elements = tf.strings.split(label_line)
    classe = tf.strings.to_number(elements[0], out_type=tf.int32)
    bbox = tf.strings.to_number(elements[1:5], out_type=tf.float32)
    return classe, bbox

def load_image_and_label(image_files, labels_files):

  classes = []
  bboxs = []
  images = []

  for label_file in labels_files:
    with open(label_file, 'r') as f:
        label_content = f.readlines()
        for line in label_content:
            classe, bbox = load_label(line)
            classes.append(classe.numpy())
            bboxs.append(bbox.numpy())

  for image_path in image_files:
      image = load_image(image_path.numpy().decode('utf-8'))
      images.append(image)

  images = tf.data.Dataset.from_tensor_slices(images)

  return images, classes, bboxs

In [ ]:
images, classes, bboxs = load_image_and_label(image_files, labels_files)
num_classes = len(set(classes))

bboxs = tf.cast(bboxs, dtype=tf.float32)
classes =  tf.cast(classes, dtype=tf.float32)

labels = tf.data.Dataset.from_tensor_slices((classes, bboxs))
dataset = tf.data.Dataset.zip((images, labels))

In [ ]:
# classes = []
# bboxs = []
# images = []

# for label_file in labels_files:
#     with open(label_file, 'r') as f:
#         label_content = f.readlines()
#         for line in label_content:
#             classe, bbox = load_label(line)
#             classes.append(classe.numpy())
#             bboxs.append(bbox)

# for image_path in image_files:
#     image = load_image(image_path.numpy().decode('utf-8'))
#     images.append(image)

# num_classes = len(set(classes))
# bboxs, classes = tf.cast(bboxs, dtype=tf.float32), tf.cast(classes, dtype=tf.float32)
# images = tf.data.Dataset.from_tensor_slices(images)
# labels = tf.data.Dataset.from_tensor_slices((classes, bboxs))
# dataset = tf.data.Dataset.zip((images, labels))

In [ ]:
def feature_extractor(inputs):
    mobilenet_model = tf.keras.applications.mobilenet_v2.MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights='imagenet',
        name="feature_extractor")

    feature_extractor = mobilenet_model(inputs)

    return feature_extractor

def build_bounding_box_layer(num_classes):
    """Builds a layer to predict bounding boxes and class probabilities."""

    def layer(inputs):
        # Convolutional layers to further extract features
        x = tf.keras.layers.Conv2D(1024, (3, 3), activation='relu', padding='same')(inputs)
        x = tf.keras.layers.MaxPooling2D((2, 2))(x)

        x = tf.keras.layers.Conv2D(512, (3, 3), activation='relu', padding='same')(x)

        # Flatten the feature maps
        x = tf.keras.layers.Flatten()(x)

        # Fully connected layer for class prediction
        class_predictions = tf.keras.layers.Dense(num_classes,
                                                  activation='softmax',
                                                  name='class_output')(x)

        bbox_predictions = tf.keras.layers.Dense(4, activation='linear',
                                                 name='bbox_output')(x)

        return class_predictions, bbox_predictions

    return layer

In [ ]:
inputs = tf.keras.layers.Input(shape=(224, 224, 3), name="Image")
feature_cnn = feature_extractor(inputs)

bounding_box_layer = build_bounding_box_layer(num_classes)
class_outputs, bbox_outputs = bounding_box_layer(feature_cnn)
model = tf.keras.Model(inputs=inputs, outputs=[class_outputs, bbox_outputs])
tf.keras.utils.plot_model(model, show_shapes=True, show_layer_names=True)

In [ ]:
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Image (InputLayer)        │ (None, 224, 224, 3)    │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ feature_extractor         │ (None, 7, 7, 1280)     │      2,257,984 │ Image[0][0]            │
│ (Functional)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_4 (Conv2D)         │ (None, 7, 7, 1024)     │     11,797,504 │ feature_extractor[0][… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ max_pooling2d_2           │ (None, 3, 3, 1024)     │              0 │ conv2d_4[0][0]         │
│ (MaxPooling2D)            │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_5 (Conv2D)         │ (None, 3, 3, 512)      │      4,719,104 │ max_pooling2d_2[0][0]  │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ flatten_2 (Flatten)       │ (None, 4608)           │              0 │ conv2d_5[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ class_output (Dense)      │ (None, 262)            │      1,207,558 │ flatten_2[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ bbox_output (Dense)       │ (None, 4)              │         18,436 │ flatten_2[0][0]        │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 20,000,586 (76.30 MB)

 Trainable params: 19,966,474 (76.17 MB)

 Non-trainable params: 34,112 (133.25 KB)

In [ ]:
def iou_loss(pred_box, true_box):
    true_box = tf.expand_dims(true_box, axis=0)
    xmin_pred, ymin_pred, xmax_pred, ymax_pred = tf.split(pred_box, 4, axis=-1)
    xmin_true, ymin_true, xmax_true, ymax_true = tf.split(true_box, 4, axis=-1)

    xmin_overlap = tf.maximum(xmin_pred, xmin_true)
    xmax_overlap = tf.minimum(xmax_pred, xmax_true)
    ymin_overlap = tf.maximum(ymin_pred, ymin_true)
    ymax_overlap = tf.minimum(ymax_pred, ymax_true)

    pred_box_area = (xmax_pred - xmin_pred) * (ymax_pred - ymin_pred)
    true_box_area = (xmax_true - xmin_true) * (ymax_true - ymin_true)

    overlap_area = tf.maximum((xmax_overlap - xmin_overlap), 0) * tf.maximum((ymax_overlap - ymin_overlap), 0)
    union_area = (pred_box_area + true_box_area) - overlap_area

    smoothing_factor = 1e-10

    iou = (overlap_area + smoothing_factor) / (union_area + smoothing_factor)

    return 1 - iou

In [ ]:
metric_bbox = tf.keras.metrics.MeanAbsoluteError()
metric_class = tf.keras.metrics.SparseCategoricalAccuracy()
optimizer = tf.keras.optimizers.SGD(learning_rate=0.001, momentum=0.9)

model.compile(optimizer=optimizer,
              loss={'class_output': 'sparse_categorical_crossentropy',
                    'bbox_output': iou_loss},
              metrics={'class_output': metric_class,
                       'bbox_output': metric_bbox})

In [ ]:
@tf.function
def train_step(images, labels, bbox):
    images = tf.expand_dims(images, axis=0)
    with tf.GradientTape() as tape:
        predictions = model(images, training=True)
        class_predictions, bbox_predictions = predictions

        num_classes = class_predictions.shape[-1]

        labels = tf.math.mod(labels, num_classes)
        labels = tf.reshape(labels, [-1])

        class_loss = tf.keras.losses.sparse_categorical_crossentropy(labels, class_predictions)
        class_loss = tf.reduce_mean(class_loss)
        bbox_loss = tf.reduce_mean(iou_loss(bbox_predictions, bbox))

        total_loss = class_loss + bbox_loss + 0.5

    gradients = tape.gradient(total_loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    return class_loss, bbox_loss

In [ ]:
EPOCHS = 50


for epoch in trange(EPOCHS):

    epoch_bbox_loss = 0

    epoch_class_loss = 0

    batch_count = 0

    for images, (labels, bbox) in dataset:

        class_loss, bbox_loss = train_step(images, labels, bbox)

        epoch_bbox_loss += bbox_loss.numpy()

        epoch_class_loss += class_loss.numpy()

        batch_count += 1

    avg_bbox_loss = epoch_bbox_loss / batch_count

    avg_class_loss = epoch_class_loss / batch_count

    print(f"Epoch {epoch+1}/{EPOCHS} Avg Class Loss: {avg_class_loss :.3f} Avg BBox Loss: {avg_bbox_loss :.3f}")

In [ ]:
def predict(image):
    # Redimensiona a imagem para o tamanho esperado pelo modelo
    image = tf.image.resize(image, [224, 224])
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.expand_dims(image, axis=0)

    # Faz as previsões
    predictions = model(image, training=False)
    class_predictions, bbox_predictions = predictions

    # Remove a dimensão extra do batch
    class_predictions = tf.squeeze(class_predictions, axis=0)
    bbox_predictions = tf.squeeze(bbox_predictions, axis=0)

    if len(bbox_predictions.shape) == 1:
        bbox_predictions = tf.reshape(bbox_predictions, [-1, 4]) # Reshape to [N, 4]

    return class_predictions, bbox_predictions

def draw_bounding_box_on_image(image, ymin, xmin, ymax, xmax, color=(255, 0, 0), thickness=5):
    """
    Desenha uma bounding box em uma imagem.

    Args:
      image: array numpy da imagem.
      ymin: ymin da bounding box (normalizado).
      xmin: xmin da bounding box (normalizado).
      ymax: ymax da bounding box (normalizado).
      xmax: xmax da bounding box (normalizado).
      color: cor da bounding box (RGB). Padrão é vermelho.
      thickness: espessura da linha. Padrão é 5.
    """
    image_height, image_width, _ = image.shape

    # Converte coordenadas normalizadas para coordenadas em pixels
    (left, right, top, bottom) = (int(xmin * image_width), int(xmax * image_width),
                                  int(ymin * image_height), int(ymax * image_height))

    # Desenha o retângulo da bounding box
    cv2.rectangle(image, (left, top), (right, bottom), color, thickness)

def draw_bounding_boxes_on_image(image, boxes, colors=None, thickness=5):
    """
    Desenha várias bounding boxes em uma imagem.

    Args:
      image: array numpy da imagem.
      boxes: array numpy [N, 4] com coordenadas das bounding boxes (normalizadas).
      colors: lista de cores (uma para cada box). Padrão é vermelho.
      thickness: espessura da linha. Padrão é 5.
    """
    if not boxes.shape[0]:
        return

    if colors is None:
        colors = [(255, 0, 0)] * boxes.shape[0]

    for i in range(boxes.shape[0]):
        color = colors[i] if i < len(colors) else (255, 0, 0)
        draw_bounding_box_on_image(image, boxes[i, 0], boxes[i, 1], boxes[i, 2], boxes[i, 3], color, thickness)

def draw_bounding_boxes_on_image_array(image, boxes, colors=None, thickness=5):
    """
    Desenha bounding boxes em uma imagem representada como array NumPy.

    Args:
      image: array numpy da imagem.
      boxes: array numpy [N, 4] com coordenadas das bounding boxes (normalizadas).
      colors: lista de cores (uma para cada box). Padrão é vermelho.
      thickness: espessura da linha. Padrão é 5.

    Returns:
      image: a imagem com as bounding boxes desenhadas.
    """
    draw_bounding_boxes_on_image(image, boxes, colors, thickness)
    return image

def plot_predictions(image_path):
    # Lê a imagem usando OpenCV
    image = cv2.imread(image_path)

    # Converte a imagem para RGB (já que o OpenCV lê como BGR)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Normaliza a imagem para estar entre 0 e 1
    image = tf.cast(image, tf.float32) / 255.0

    # Previsões de classes e caixas
    class_predictions, boxes = predict(image)

    # Garante que a imagem está de volta em formato NumPy
    image = image.numpy()

    # Desenha as bounding boxes na imagem
    draw_bounding_boxes_on_image_array(image, boxes, colors=[(0, 255, 0)], thickness=5)

    # Exibe a imagem resultante
    plt.imshow(image)
    plt.axis('off')
    plt.show()

In [ ]:
images_path = glob.glob("/content/test/images/*")

for image_path in images_path:
  plot_predictions(image_path)